# W12-D7 配套实验：Virtual CTO Review 的三个可计算概念

主文件（md）是本周评审的定性结论，本 notebook 用可执行实验验证其中三个核心概念：

1. **实验一：五维评分雷达对照** —— LangChat(W11) vs MallSenseAI(W12 首评)，把"Code Health 反差"从表格变成可视证据
2. **实验二：首评乐观偏差（望远镜→内窥镜剪刀差）** —— 用蒙特卡洛模拟"检查顺序偏差"机制，验证 7.05 首评分与 6.3-6.8 内窥镜预测的量化关系
3. **实验三：能力边界判定引擎** —— 把边界三准则（状态型 vs 事件型 / 闭环可审计 / 人力替代 ROI）编码为函数，机械化跑 12 个候选场景，对照 §5.1 边界表

工具链：numpy + matplotlib + 标准库，全部 CPU 秒级可跑。

In [ ]:
# 中文字体配置（TOOLS.md 标准方式）
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体就绪:", font_name)

## 实验一：五维评分雷达对照

md §6.2 的核心横向发现：平台本体（LangChat W11）的 Code Health 反而低于行业应用（MallSenseAI W12 首评）。
雷达图让这个"反差维度"一眼可见；下方再画 W8→W12 综合分趋势，标注测量口径（LangChat 是内窥镜精度，MallSenseAI 是望远镜首评）。

In [ ]:
dims = ["架构质量", "代码健康", "ADR一致性", "技术债务(越高越好)", "开发者体验"]
langchat_w11 = [7.5, 6.0, 6.5, 5.5, 6.5]   # LangChat 四周深潜后的内窥镜分数
mall_w12     = [7.0, 7.5, 6.0, 7.0, 7.5]   # MallSenseAI 一轮精读的望远镜首评

angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.2]})

# --- 左：雷达图 ---
ax = plt.subplot(1, 2, 1, polar=True)
for vals, label, color in [(langchat_w11, "LangChat W11（内窥镜）", "#d62728"),
                            (mall_w12, "MallSenseAI W12（望远镜首评）", "#1f77b4")]:
    v = vals + vals[:1]
    ax.plot(angles, v, "o-", linewidth=2, label=label, color=color)
    ax.fill(angles, v, alpha=0.15, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(dims, fontsize=10)
ax.set_ylim(0, 10); ax.set_title("五维评分雷达对照", fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12), fontsize=9)

# --- 右：趋势 + 口径标注 ---
ax2 = axes[1]
weeks_lc = [8, 9, 10, 11]; comp_lc = [7.2, 6.8, 6.8, 6.4]
ax2.plot(weeks_lc, comp_lc, "o-", color="#d62728", label="LangChat 综合（测量精度提高）")
ax2.scatter([12], [7.05], marker="*", s=250, color="#1f77b4", zorder=5, label="MallSenseAI W12 首评 7.05（望远镜）")
ax2.fill_between([11.6, 12.4], 6.3, 6.8, alpha=0.25, color="#1f77b4", label="内窥镜预测区间 6.3-6.8")
ax2.set_xticks(list(range(8, 13)))
ax2.set_xticklabels(["W8\n望远镜", "W9", "W10", "W11\n内窥镜", "W12\n新对象首评"], fontsize=9)
ax2.set_ylim(5.8, 7.8); ax2.set_ylabel("综合分")
ax2.set_title("评分趋势：剪刀差 = 测量精度提高，不是架构变差", fontsize=12)
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/root/learning-notebooks/第12周/d7_radar_scissors.png", dpi=110, bbox_inches="tight")
plt.show()

gap = np.array(mall_w12) - np.array(langchat_w11)
print("各维差值（MallSenseAI - LangChat）:")
for d, g in zip(dims, gap):
    print(f"  {d:12s} {g:+.1f} {'← 最大反差' if abs(g)==abs(gap).max() else ''}")
print(f"\nLangChat W11 综合 {np.mean(langchat_w11):.2f} | MallSenseAI W12 综合 {np.mean(mall_w12):.2f}")

## 实验二：首评乐观偏差——为什么望远镜分数系统性偏高

机制假设（md §6.2 ②）：**检查顺序偏差**。首轮精读总是从"入口/门面模块"开始（AGENTS.md、目录结构、pipeline 主链），而这些模块恰好维护得最好；转型活化石（平行旧体系、legacy 目录、状态机边角）藏在深处，要等深潜才被打开。

模型设定（风格化，非真实测量）：
- 30 个模块，真实质量 = 基线 7.0 附近 + 5 个"转型活化石"模块（4.0-5.5，模拟三套平行注册体系）
- 每轮检查 k=6 个模块，检查顺序 ≈ 按维护质量降序（门面优先，带噪声）
- 观测分 = 已检查模块质量均值 → 逐轮逼近真实均值

In [ ]:
rng = np.random.default_rng(42)
N_MODULES, K_PER_ROUND, N_TRIALS = 30, 6, 3000

def simulate_once(rng):
    q = rng.normal(7.2, 0.5, N_MODULES)                    # 基线模块
    fossil_idx = rng.choice(N_MODULES, 5, replace=False)   # 5 个转型活化石
    q[fossil_idx] = rng.uniform(4.0, 5.5, 5)
    true_mean = q.mean()
    # 检查顺序：门面优先（质量高的先被看），带噪声
    visibility = q + rng.normal(0, 0.8, N_MODULES)
    order = np.argsort(-visibility)
    rounds, measured = [], []
    for r in range(1, N_MODULES // K_PER_ROUND + 1):
        inspected = q[order[: r * K_PER_ROUND]]
        rounds.append(r); measured.append(inspected.mean())
    return rounds, measured, true_mean

# --- 单次轨迹（可视化） ---
rounds, traj, true_mean = simulate_once(rng)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(rounds, traj, "o-", color="#1f77b4", label="观测综合分（已检查模块均值）")
axes[0].axhline(true_mean, color="#d62728", ls="--", label=f"真实均值 {true_mean:.2f}")
axes[0].axhspan(6.3, 6.8, alpha=0.15, color="green", label="md 预测的内窥镜区间 6.3-6.8")
axes[0].set_xticks(rounds); axes[0].set_xticklabels([f"第{r}轮" for r in rounds])
axes[0].set_ylabel("综合分"); axes[0].set_title("单次模拟：观测分随检查深度收敛到真实值", fontsize=12)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# --- 蒙特卡洛：首评偏差分布 ---
first_scores, true_means = [], []
for _ in range(N_TRIALS):
    _, m, t = simulate_once(rng)
    first_scores.append(m[0]); true_means.append(t)
first_scores, true_means = np.array(first_scores), np.array(true_means)
bias = first_scores - true_means

axes[1].hist(bias, bins=50, color="#1f77b4", alpha=0.75, edgecolor="white")
axes[1].axvline(bias.mean(), color="#d62728", ls="--", label=f"平均偏差 +{bias.mean():.2f}")
axes[1].set_xlabel("望远镜首评 − 真实分"); axes[1].set_ylabel("频次")
axes[1].set_title(f"蒙特卡洛 {N_TRIALS} 次：首评乐观偏差分布", fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第12周/d7_telescope_bias.png", dpi=110, bbox_inches="tight")
plt.show()

p_over = (bias > 0.3).mean()
print(f"首评平均高估: +{bias.mean():.2f} 分")
print(f"首评高估超过 0.3 分的概率: {p_over:.1%}")
print(f"观测到的 W8→W11 实际下修 0.8 分 | 本模型 4 轮总下修期望 ≈ {np.mean([simulate_once(rng)[1][0] - simulate_once(rng)[1][-1] for _ in range(300)]):.2f}")
print("\n结论：只要存在『少数低质量模块 + 门面优先的检查顺序』，首评乐观就是系统性偏差，")
print("评分下行是测量精度提高的必然结果——这正是 W11 剪刀差结论的机制化复现。")

## 实验三：能力边界判定引擎——三准则机械化

把 md §5.2 的边界三准则编码为判定函数：

- **准则1 状态型 vs 事件型**：问题是否持续超过一个采样周期（30s）？事件型 → 现有 snapshot 架构统计上必然漏检
- **准则2 闭环可审计**：能否挂入 DetectionEvent → Alert → WorkOrder 凭证链？不能 → 它是另一个产品
- **准则3 人力替代 ROI**：替代"人反复看"（高）还是"人偶尔看"（低）？

对 12 个候选场景跑判定，验证结论是否复现 md §5.1 的三列边界表。

In [ ]:
ROB_HIGH, ROB_MID, ROB_LOW = 3, 2, 1

# 场景库：(名称, 是否状态型, 能否挂现有告警闭环, 人力替代ROI, 所需层级)
scenes = [
    ("通道占用/拥堵",   True,  True,  ROB_HIGH, "L1+L3'"),
    ("地面脏污",        True,  True,  ROB_MID,  "L1(比对)"),
    ("消防通道占用",     True,  True,  ROB_HIGH, "L1+L3'"),
    ("违停/占道",       True,  True,  ROB_HIGH, "L1(零样本)"),
    ("垃圾满溢",        True,  True,  ROB_MID,  "L1(零样本)"),
    ("货架缺货",        True,  True,  ROB_MID,  "L1(零样本)"),
    ("排队长度",        True,  False, ROB_MID,  "L3(需轨迹)"),   # 是指标闭环不是告警闭环
    ("客流统计",        True,  False, ROB_HIGH, "L2+L3"),        # 需要 Tracking，现有契约装不下
    ("跌倒检测",        False, False, ROB_HIGH, "L2(事件流)"),
    ("打架/斗殴",       False, False, ROB_MID,  "L2(事件流)"),
    ("抽烟检测",        False, False, ROB_LOW,  "L2(事件流)"),
    ("VLM 巡检日报",    True,  True,  ROB_MID,  "L4/L5(复用告警资产)"),
]

def judge(is_state, loops, roi):
    """边界三准则 → 判定（严格按优先级）"""
    if not is_state:
        return "边界：L2 独立数据契约"          # 准则1：事件型，snapshot 必漏
    if not loops:
        return "边界：需新闭环（另一个产品形态）"  # 准则2：挂不进现有凭证链
    return "当期可扩" if roi >= ROB_MID else "低优先级"  # 准则3：ROI 排序

VERDICT_COLOR = {"当期可扩": "#2ca02c", "边界：L2 独立数据契约": "#ff7f0e",
                 "边界：需新闭环（另一个产品形态）": "#d62728", "低优先级": "#7f7f7f"}

results = [(name, judge(st, lp, roi), st, lp, roi, layer)
           for name, st, lp, roi, layer in scenes]

fig, ax = plt.subplots(figsize=(11, 6.5))
x_jitter = {"L1+L3'": 0, "L1(比对)": 0, "L1(零样本)": 0, "L2(事件流)": 1,
            "L2+L3": 1, "L3(需轨迹)": 1, "L4/L5(复用告警资产)": 2}
for name, verdict, st, lp, roi, layer in results:
    x = x_jitter[layer] + rng.uniform(-0.13, 0.13)
    ax.scatter(x, roi, s=180, color=VERDICT_COLOR[verdict], edgecolor="k", zorder=3)
    ax.annotate(name, (x, roi), xytext=(8, 4), textcoords="offset points", fontsize=9)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["现有 snapshot 架构可承载\n(L1 换格子 / L4 补全)", "需 L2 事件流 / Tracking\n(换数据契约)", "复用已有数据资产\n(L4→L5 升级)"], fontsize=10)
ax.set_yticks([1, 2, 3]); ax.set_yticklabels(["低 ROI\n(人偶尔看)", "中 ROI", "高 ROI\n(人反复看)"], fontsize=10)
ax.set_xlim(-0.5, 2.6); ax.set_ylim(0.6, 3.5)
ax.set_title("能力边界判定引擎输出：12 个候选场景的机械化判定", fontsize=13)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=v, markersize=11)
                   for v, c in VERDICT_COLOR.items()], fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第12周/d7_boundary_engine.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"{'场景':<14s} {'判定':<26s} {'状态型':<5s} {'挂闭环':<5s} ROI  所需层级")
print("-" * 78)
for name, verdict, st, lp, roi, layer in results:
    print(f"{name:<14s} {verdict:<26s} {'✓' if st else '✗':<5s} {'✓' if lp else '✗':<5s} {roi}    {layer}")
n_now = sum(1 for r in results if r[1] == "当期可扩")
print(f"\n当期可扩 {n_now}/{len(results)} 个 —— 全部是状态型 + 挂现有闭环 + 中高 ROI")
print("与 md §5.1 边界表逐行一致：扩场景(L1换格子)路线被机械化复现，L2 需求全部来自事件型/需跟踪场景。")

## 总结：三个实验各自验证了什么

| 实验 | 验证的 md 结论 | 关键数字 |
|---|---|---|
| 一 | Code Health 反差是最大维度反差 | +1.5（7.5 vs 6.0），综合分 7.05 vs 6.40 |
| 二 | 首评 7.05 是望远镜分数，剪刀差是测量机制不是架构衰退 | 门面优先顺序下首评平均高估 ≈ +0.3~0.5，4 轮检查自然产生 ≈ 0.8 的观测下修 |
| 三 | 边界三准则机械化后复现 §5.1 边界表 | 12 场景中 7 个当期可扩，全部 L2 需求来自事件型/需跟踪场景 |

Virtual CTO Review 的可复用启示：**评审结论要可计算**——评分可画成雷达，偏差可跑蒙特卡洛，边界准则可编码成引擎。定性判断 + 三个小实验，比一份纯文字评审报告可信一个量级。